# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [81]:
# Challenge 1
import pandas as pd
url_1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url_2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url_3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

# load data
df1 = pd.read_csv(url_1)
df2 = pd.read_csv(url_2)
df3 = pd.read_csv(url_3)

# create list of dfs
dfs = [df1, df2, df3]

# check columns and shape for all dfs
for df in dfs:
    print (df.columns, df.shape)

# fix column names in df1 and df2
df1 = df1.rename(columns={"ST": "State", "GENDER": "Gender"})
df2 = df2.rename(columns={"ST": "State", "GENDER": "Gender"})

# concat dfs
df = pd.concat([df1, df2, df3], ignore_index=True)

print(df.shape)
df.head()

Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Vehicle Class', 'Total Claim Amount'],
      dtype='object') (4008, 11)
Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Total Claim Amount', 'Policy Type', 'Vehicle Class'],
      dtype='object') (996, 11)
Index(['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Total Claim Amount', 'Vehicle Class'],
      dtype='object') (7070, 11)
(12074, 11)


,Customer,State,Gender,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [82]:
# cleaning of combined df
# fix column names
df.columns = df.columns.str.replace(' ', '_').str.lower()

# check for rows with all NaN
initial_shape = df.shape
df = df.dropna(how='all')
final_shape = df.shape

print(f"Number of rows dropped: {initial_shape[0] - final_shape[0]}")

# check for dublicates
df.duplicated().any()
df = df.drop_duplicates().reset_index(drop=True)

# check if recoding needs to be done
for col in df.columns:
    if df[col].dtype == 'object':
        print(f'{col}:', '\n', f'* Nr. of unique vals: {(df[col].nunique())}', '\n', f'* unique vals: {df[col].unique()}')
        print("\n")

Number of rows dropped: 2937
customer: 
 * Nr. of unique vals: 9056 
 * unique vals: ['RB50392' 'QZ44356' 'AI49188' ... 'TD14365' 'UP19263' 'Y167826']


state: 
 * Nr. of unique vals: 8 
 * unique vals: ['Washington' 'Arizona' 'Nevada' 'California' 'Oregon' 'Cali' 'AZ' 'WA']


gender: 
 * Nr. of unique vals: 5 
 * unique vals: [nan 'F' 'M' 'Femal' 'Male' 'female']


education: 
 * Nr. of unique vals: 6 
 * unique vals: ['Master' 'Bachelor' 'High School or Below' 'College' 'Bachelors' 'Doctor']


customer_lifetime_value: 
 * Nr. of unique vals: 8211 
 * unique vals: [nan '697953.59%' '1288743.17%' ... 8163.890428 7524.442436 2611.836866]


number_of_open_complaints: 
 * Nr. of unique vals: 12 
 * unique vals: ['1/0/00' '1/2/00' '1/1/00' '1/3/00' '1/5/00' '1/4/00' 0 2 3 1 5 4]


policy_type: 
 * Nr. of unique vals: 3 
 * unique vals: ['Personal Auto' 'Corporate Auto' 'Special Auto']


vehicle_class: 
 * Nr. of unique vals: 6 
 * unique vals: ['Four-Door Car' 'Two-Door Car' 'SUV' 'Luxury 

In [85]:
# cleaning single vars

# gender
df['gender'] = df['gender'].replace(['Femal', 'female'], 'F')
df['gender'] = df['gender'].replace('Male', 'M')

# state
df['state'] = df['state'].replace({'AZ': 'Arizona', 'WA': 'Washington', 'Cali': 'California'})

# edu
df['education'] = df['education'].replace('Bachelors', 'Bachelor')

# CLV
df['customer_lifetime_value'] = df['customer_lifetime_value'].str.strip('%').apply(lambda x: float(x))

# vehicle class
df['vehicle_class'] = df['vehicle_class'].replace(["Sports Car", "Luxury SUV", "Luxury Car"],'Luxury')

# complaints
df['number_of_open_complaints'] = df['number_of_open_complaints'].str.split('/').str[1]
df['number_of_open_complaints'] = df['number_of_open_complaints'].astype(float)

In [87]:
# check if recoding worked and if I overlooked something
for col in df.columns:
    if df[col].dtype == 'object':
        print(f'{col}:', '\n', f'* Nr. of unique vals: {(df[col].nunique())}', '\n', f'* unique vals: {df[col].unique()}')
        print("\n")

customer: 
 * Nr. of unique vals: 9056 
 * unique vals: ['RB50392' 'QZ44356' 'AI49188' ... 'TD14365' 'UP19263' 'Y167826']


state: 
 * Nr. of unique vals: 5 
 * unique vals: ['Washington' 'Arizona' 'Nevada' 'California' 'Oregon']


gender: 
 * Nr. of unique vals: 2 
 * unique vals: [nan 'F' 'M']


education: 
 * Nr. of unique vals: 5 
 * unique vals: ['Master' 'Bachelor' 'High School or Below' 'College' 'Doctor']


policy_type: 
 * Nr. of unique vals: 3 
 * unique vals: ['Personal Auto' 'Corporate Auto' 'Special Auto']


vehicle_class: 
 * Nr. of unique vals: 4 
 * unique vals: ['Four-Door Car' 'Two-Door Car' 'SUV' 'Luxury']




# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [107]:
# Challenge 2
import pandas as pd
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv'
df = pd.read_csv(url)

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [109]:
# 1.
pivot_df = df.pivot_table(index='sales_channel', values='total_claim_amount', aggfunc='sum').round(2)
pivot_df

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [115]:
# 2.
# Pivot the DataFrame to see the GDP based on the country and year
pivot_df = df.pivot_table(index='education', columns='gender', values=['customer_lifetime_value'], aggfunc='mean')
pivot_df

customer_lifetime_value             
gender                                     F            M
education                                                
Bachelor                         7874.269478  7703.601675
College                          7748.823325  8052.459288
Doctor                           7328.508916  7415.333638
High School or Below             8675.220201  8149.687783
Master                           8157.053154  8168.832659

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here